In [1]:
import pandas as pd
from IPython.display import display
from pulp import LpProblem, LpVariable, lpSum, LpMaximize, LpBinary, PULP_CBC_CMD
from helpers import CLASSES, ITEM_CLASS_MAP, CLASS_PRIMARY_ATTR

pd.set_option('display.max_rows', 25)

In [2]:
def get_primary_stat_for_class(hero_class, class_primary_attr):
    for attr, class_list in class_primary_attr.items():
        if hero_class in class_list:
            return attr
    raise ValueError(f"Primary attribute not found for class '{hero_class}'!")

def get_allowed_classes(item_class_value, item_class_map):
    allowed = set()
    for class_group in str(item_class_value).split(":"):
        class_group = class_group.strip()
        allowed.update(item_class_map.get(class_group, []))
    return allowed

def filter_items(df, hero_class, dungeon_ready, item_class_map):
    def can_wield(row):
        allowed = get_allowed_classes(row['Class'], item_class_map)
        return (hero_class in allowed) and (row['Dungeon'] <= dungeon_ready)
    return df[df.apply(can_wield, axis=1)].reset_index(drop=True)

def build_stat_cols(primary_stat, stat_cols):
    dmg_col = f'Total_DMG_{primary_stat}'
    reordered = [dmg_col] + [col for col in stat_cols if col != dmg_col]
    return reordered

def set_primary_damage_weights(weights, stat_cols, primary_stat):
    dmg_stats = ["Total_DMG_Str", "Total_DMG_Agi", "Total_DMG_Int"]
    for dmg in dmg_stats:
        if dmg != f"Total_DMG_{primary_stat}" and dmg in stat_cols:
            idx = stat_cols.index(dmg)
            weights[idx] = 0
    return weights

def only_one_crit_item(x_vars, stats, stat_cols):
    cc_idx = stat_cols.index("CC") if "CC" in stat_cols else None
    cx_idx = stat_cols.index("CX") if "CX" in stat_cols else None
    crit_indices = set()
    if cc_idx is not None:
        crit_indices.update([i for i in range(len(x_vars)) if stats[i, cc_idx] > 0])
    if cx_idx is not None:
        crit_indices.update([i for i in range(len(x_vars)) if stats[i, cx_idx] > 0])
    if not crit_indices:
        return 0
    return lpSum([x_vars[i] for i in crit_indices]) <= 1

def at_least_one_mr_if_possible(x_vars, stats, stat_cols):
    if "MR" not in stat_cols:
        return 0
    mr_idx = stat_cols.index("MR")
    mr_indices = [i for i in range(len(x_vars)) if stats[i, mr_idx] > 0]
    if not mr_indices:
        return 0
    return lpSum([x_vars[i] for i in mr_indices]) >= 1

def at_least_one_cdr_if_possible(x_vars, stats, stat_cols):
    if "CDS" not in stat_cols:
        return 0
    cdr_idx = stat_cols.index("CDS")
    cdr_indices = [i for i in range(len(x_vars)) if stats[i, cdr_idx] > 0]
    if not cdr_indices:
        return 0
    return lpSum([x_vars[i] for i in cdr_indices]) >= 1

def optimize_hero_build(
    items_csv_path,
    weights_csv_path,
    hero_class,
    dungeon_ready,
    item_limit=6,
    item_class_map=None,
    class_primary_attr=None,
    custom_constraints=None
):
    items_df = pd.read_csv(items_csv_path).fillna(0)
    weights_df = pd.read_csv(weights_csv_path, index_col=0)

    primary_stat = get_primary_stat_for_class(hero_class, class_primary_attr)

    stat_cols = [col for col in weights_df.columns if col in items_df.columns]
    stat_cols = build_stat_cols(primary_stat, stat_cols)

    filtered_items = filter_items(items_df, hero_class, dungeon_ready, item_class_map)
    print(f"Filtered items: {filtered_items.shape[0]} available for class {hero_class}")

    if filtered_items.empty:
        print("No items available after filtering.")
        return pd.DataFrame(), pd.Series()

    if item_limit > len(filtered_items):
        raise ValueError(
            f"Not enough items to satisfy item_limit={item_limit}. Only {len(filtered_items)} items available."
        )

    try:
        weights = weights_df.loc[hero_class, stat_cols].values
    except KeyError:
        raise ValueError(f"Class '{hero_class}' not found in weights CSV.")

    weights = set_primary_damage_weights(weights, stat_cols, primary_stat)
    stats = filtered_items[stat_cols].values
    n_items = len(filtered_items)

    prob = LpProblem("RPG_Best_Item_Combo", LpMaximize)
    x_vars = [LpVariable(f"x_{i}", 0, 1, LpBinary) for i in range(n_items)]
    total_stats = [
        lpSum([stats[i, j] * x_vars[i] for i in range(n_items)])
        for j in range(len(stat_cols))
    ]
    prob += lpSum([weights[j] * total_stats[j] for j in range(len(weights))]), "TotalWeightedStats"
    prob += lpSum(x_vars) == item_limit, "ItemLimit"

    if custom_constraints:
        for fn in custom_constraints:
            prob += fn(x_vars, stats, stat_cols)

    prob.solve(PULP_CBC_CMD(msg=0))
    selected_indices = [i for i, var in enumerate(x_vars) if var.varValue is not None and var.varValue > 0.5]

    if not selected_indices:
        print("No solution found: selected_indices is empty.")
        return pd.DataFrame(), pd.Series()

    # Columns to always display, if present
    RAW_STATS = ["DMG", "HP", "HPR", "MP", "MPR", "Armor"]

    # Optimization stats (from weights)
    primary_dmg_col = f"Total_DMG_{primary_stat}"
    opt_stats = [
        primary_dmg_col
    ] + [
        col for col in stat_cols
        if col not in (primary_dmg_col, "Item", "CC", "CX")
        and not col.startswith("Total_DMG_")
    ]
    if "CC" in stat_cols:
        opt_stats.append("CC")
    if "CX" in stat_cols:
        opt_stats.append("CX")

    # Combine columns, but only include those present in filtered_items
    display_cols = ["Item"] + [col for col in RAW_STATS if col in filtered_items.columns] + [col for col in opt_stats if col in filtered_items.columns and col not in RAW_STATS]

    selected_items = filtered_items.iloc[selected_indices][display_cols]

    # Calculate totals for all display columns
    total_dict = {}
    for col in display_cols:
        if col == "Item":
            continue
        total_dict[col] = selected_items[col].sum() if col in selected_items else 0
    total_stats_series = pd.Series(total_dict)

    # Weighted stats only for those in weights (matching total_stats)
    weights_dict = dict(zip(
        [col for col in total_stats_series.index if col in weights_df.columns],
        weights_df.loc[hero_class, [col for col in total_stats_series.index if col in weights_df.columns]]
    ))
    weighted_stats = total_stats_series.copy()
    for col in weighted_stats.index:
        if col in weights_dict:
            weighted_stats[col] = weighted_stats[col] * weights_dict[col]
        else:
            weighted_stats[col] = None  # Or zero if you prefer

    # Combine total and weighted stats side by side
    stats_df = pd.DataFrame({
        'Total': total_stats_series,
        'Weighted': weighted_stats
    })

    return selected_items, stats_df

In [21]:
# Example usage
ITEMS_CSV = "items.csv"
WEIGHTS_CSV = "class_weights.csv"
HERO_CLASS = "Dark ArchTemplar"
DUNGEON_READY = 11
ITEM_LIMIT = 6

CUSTOM_CONSTRAINTS = [
    # at_least_one_cdr_if_possible,
    only_one_crit_item,
    # at_least_one_mr_if_possible
]

selected_items, stats_df = optimize_hero_build(
    items_csv_path=ITEMS_CSV,
    weights_csv_path=WEIGHTS_CSV,
    hero_class=HERO_CLASS,
    dungeon_ready=DUNGEON_READY,
    item_limit=ITEM_LIMIT,
    item_class_map=ITEM_CLASS_MAP,
    class_primary_attr=CLASS_PRIMARY_ATTR,
    custom_constraints=CUSTOM_CONSTRAINTS
)


print("\n=== Optimized Build ===")
display(selected_items)

print("\n=== Total Stats ===")
display(stats_df)

Filtered items: 65 available for class Dark ArchTemplar

=== Optimized Build ===


,Item,DMG,HP,HPR,MP,MPR,Armor,Total_DMG_Agi,Str,Agi,...,CDS,MR,MS,Total_HP,Total_HPR,Total_Armor,Total_MP,Total_MPR,CC,CX
49,Moonfang,2000.0,0.0,0.0,0.0,0.0,0.0,2400.0,0.0,400.0,...,0.00,0.0,0.0,0.0,0.0,33.0,0.0,0.0,0.0,0.0
51,Blade of the Ruined King,3600.0,7000.0,0.0,0.0,0.0,0.0,3600.0,0.0,0.0,...,0.00,0.0,0.0,7000.0,0.0,0.0,0.0,0.0,0.0,0.0
57,Crystallized Emerald Blade,2500.0,0.0,0.0,0.0,0.0,0.0,2750.0,250.0,250.0,...,0.00,0.0,0.0,5000.0,13.0,21.0,3750.0,13.0,0.0,0.0
58,Doombringer,0.0,0.0,0.0,0.0,0.0,0.0,850.0,0.0,850.0,...,0.05,0.0,0.0,0.0,0.0,71.0,0.0,0.0,0.0,0.0
61,Phantom Dancer,0.0,0.0,0.0,0.0,0.0,0.0,500.0,0.0,500.0,...,0.00,0.0,0.0,0.0,0.0,42.0,0.0,0.0,0.0,0.0
64,Tomahawk Axe,4800.0,0.0,0.0,0.0,0.0,0.0,4800.0,0.0,0.0,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



=== Total Stats ===


,Total,Weighted
DMG,12900.00,NaN
HP,7000.00,NaN
HPR,0.00,NaN
MP,0.00,NaN
MPR,0.00,NaN
Armor,0.00,NaN
Total_DMG_Agi,14900.00,149000.0
Str,250.00,0.0
Agi,2000.00,100000.0
Int,250.00,0.0
